In [41]:
!pip install fastcoref==2.1.1 -q
!pip install spacy
!pip install sentence_transformers

In [42]:
from fastcoref import FCoref
import spacy
from dataclasses import dataclass
import json
import re
import subprocess
from sentence_transformers import SentenceTransformer, util
import pandas as pd

In [43]:
# entity extraction (part from the script to build graphs)

@dataclass
class Entity:
    id: int
    text: str

@dataclass
class EntityMention:
    entity: Entity
    word_span: tuple[int, int]   # (start_word, end_word)
    char_span: tuple[int, int]   # (start_char, end_char)

@dataclass
class Document:
    text: str
    entities: list[Entity]
    entity_mentions: list[EntityMention]

@dataclass
class Paragraph:
    text: str
    entity_mentions: list[EntityMention]

@dataclass
class Relation:
    head: int
    relation: str
    tail: int


def _mark_entities(paragraph: Paragraph) -> str:

    marked = paragraph.text

    mentions = sorted(
        paragraph.entity_mentions,
        key=lambda m: m.char_span[0],
        reverse=True
    )

    for mention in mentions:
        start, end = mention.char_span

        marked = (
            marked[:start]
            + f'<entity id="{mention.entity.id}">'
            + marked[start:end]
            + "</entity>"
            + marked[end:]
        )

    return marked


def _split_into_paragraphs(document: Document) -> list[Paragraph]:

    paragraphs = []
    text = document.text
    offset = 0

    for paragraph_text in text.split("\n\n"):

        paragraph_start = offset
        paragraph_end = paragraph_start + len(paragraph_text)

        mentions = []
        for mention in document.entity_mentions:
            start, end = mention.char_span
            if paragraph_start <= start and end <= paragraph_end:

                mentions.append(
                    EntityMention(
                        entity=mention.entity,
                        word_span=mention.word_span,
                        char_span=(
                            start - paragraph_start,
                            end - paragraph_start
                        )
                    )
                )

        paragraphs.append(
            Paragraph(
                text=paragraph_text,
                entity_mentions=mentions
            )
        )
        offset = paragraph_end + 2

    return paragraphs


def extract_entities(
    text: str,
    model=None,
    nlp=None
) -> Document:

    if model is None:
        model = FCoref(device="cpu")

    if nlp is None:
        nlp = spacy.load("en_core_web_sm")

    preds = model.predict(texts=[text])

    clusters = preds[0].get_clusters(as_strings=False)
    doc = nlp(text)
    result = []
    all_indices_in_clusters = set()

    for cluster_id, cluster in enumerate(clusters):
        for start_char, end_char in cluster:
            span = doc.char_span(
                start_char,
                end_char,
                alignment_mode="expand"
            )

            if span is None:
                continue

            indices = tuple(token.i for token in span)
            result.append({
                "cluster_id": cluster_id,
                "text": span.text,
                "indices": indices,
                "char_span": (start_char, end_char)
            })
            all_indices_in_clusters.update(indices)

    ner_entities = []
    for ent in doc.ents:
        indices = tuple(token.i for token in ent)
        ner_entities.append({
            "text": ent.text,
            "indices": indices,
            "char_span": (ent.start_char, ent.end_char),
            "label": ent.label_
        })

    remaining_ner = []
    for entity in ner_entities:
        if not set(entity["indices"]).issubset(all_indices_in_clusters):
            remaining_ner.append(entity)

    next_cluster_id = len(set(item["cluster_id"] for item in result))
    for entity in remaining_ner:
        result.append({
            "cluster_id": next_cluster_id,
            "text": entity["text"],
            "indices": entity["indices"],
            "char_span": entity["char_span"]
        })
        next_cluster_id += 1

    unique_cluster_ids = sorted(
        set(item["cluster_id"] for item in result)
    )
    cluster_mapping = {
        old: new
        for new, old in enumerate(unique_cluster_ids)
    }

    entities = {}
    for item in result:
        entity_id = cluster_mapping[item["cluster_id"]]
        if entity_id not in entities:
            entities[entity_id] = Entity(
                id=entity_id,
                text=item["text"]
            )

    mentions = []
    for item in result:
        indices = item["indices"]
        if len(indices) == 0:
            continue

        mentions.append(
            EntityMention(
                entity=entities[cluster_mapping[item["cluster_id"]]],
                word_span=(indices[0], indices[-1]),
                char_span=item["char_span"]
            )
        )

    return Document(
        text=text,
        entities=list(entities.values()),
        entity_mentions=mentions
    )

In [44]:
def extract_entities_spacy(
    text: str,
    nlp=None
) -> Document:
    
    if nlp is None:
        nlp = spacy.load("en_core_web_sm")

    doc = nlp(text)

    ner_entities = []
    for ent in doc.ents:
        indices = tuple(token.i for token in ent)
        ner_entities.append({
            "text": ent.text,
            "indices": indices,
            "char_span": (ent.start_char, ent.end_char),
            "label": ent.label_
        })

    def merge_split_entities(entities):
    
        if len(entities) <= 1:
            return entities
        
        ALLOWED_LABELS = {"PERSON", "ORG", "GPE", "DATE"}
        
        merged = []
        skip_indices = set()
        
        i = 0
        while i < len(entities):
            if i in skip_indices:
                i += 1
                continue
            
            current = entities[i]
            
            if current["label"] not in ALLOWED_LABELS:
                merged.append(current)
                i += 1
                continue
            
            merged_text = current["text"]
            merged_start = current["char_span"][0]
            merged_end = current["char_span"][1]
            merged_indices = list(current["indices"])
            merged_label = current["label"]
            
            j = i + 1
            while j < len(entities):
                if j in skip_indices:
                    j += 1
                    continue
                
                next_ent = entities[j]
                
                gap = next_ent["char_span"][0] - current["char_span"][1]
                if (current["label"] == next_ent["label"] and 
                    0 <= gap <= 5 and
                    current["label"] in ALLOWED_LABELS):
                    
                    merged_text = current["text"] + " " + next_ent["text"]
                    merged_start = min(current["char_span"][0], next_ent["char_span"][0])
                    merged_end = max(current["char_span"][1], next_ent["char_span"][1])
                    merged_indices = list(current["indices"]) + list(next_ent["indices"])
                    merged_indices = sorted(set(merged_indices))
                    merged_label = current["label"]
                    
                    skip_indices.add(j)
                    current = {
                        "text": merged_text,
                        "indices": tuple(merged_indices),
                        "char_span": (merged_start, merged_end),
                        "label": merged_label
                    }
                    j += 1
                else:
                    break
            
            merged.append(current)
            i += 1
        
        return merged
    
    ner_entities = merge_split_entities(ner_entities)
    

    text_to_entities = {}
    entity_id_counter = 0

    for entity in ner_entities:
        key = entity["text"].lower()
        
        if key not in text_to_entities:
            text_to_entities[key] = {
                "entity_id": entity_id_counter,
                "text": entity["text"],
                "mentions": []
            }
            entity_id_counter += 1
        
        text_to_entities[key]["mentions"].append({
            "indices": entity["indices"],
            "char_span": entity["char_span"]
        })

    entities = {}
    for key, data in text_to_entities.items():
        entity_id = data["entity_id"]
        entities[entity_id] = Entity(
            id=entity_id,
            text=data["text"]
        )

    mentions = []
    for key, data in text_to_entities.items():
        entity_id = data["entity_id"]
        for mention_data in data["mentions"]:
            indices = mention_data["indices"]
            if len(indices) == 0:
                continue
            
            mentions.append(
                EntityMention(
                    entity=entities[entity_id],
                    word_span=(indices[0], indices[-1]),
                    char_span=mention_data["char_span"]
                )
            )

    return Document(
        text=text,
        entities=list(entities.values()),
        entity_mentions=mentions
    )

In [45]:
def prepare_comparison_data_with_coref(samples, model=None, nlp=None):

    if model is None:
        model = FCoref(device="cpu")
    
    if nlp is None:
        nlp = spacy.load("en_core_web_sm")
    
    for_comparison = []
    
    for sample in samples:
        text = sample['target']
        expected_triplets = sample['input']
        
        all_entities = set()
        found_entities = [entity.text.lower() for entity in extract_entities(text, model=model, nlp=nlp).entities]
        
        for triplet in expected_triplets:
            first_ent = re.search('^[^|]*', triplet).group().strip().lower().replace('_', ' ')
            second_ent = re.search('[^|]*$', triplet).group().strip().lower().replace('_', ' ')
            
            all_entities.add(first_ent)
            all_entities.add(second_ent)
        
        compare_pair = [text, list(all_entities), found_entities]
        for_comparison.append(compare_pair)
    
    return for_comparison

In [46]:
def prepare_comparison_data_spacy(samples, nlp=None):
    
    if nlp is None:
        nlp = spacy.load("en_core_web_sm")
    
    for_comparison_spacy = []
    
    for sample in samples:
        text = sample['target']
        expected_triplets = sample['input']
        
        all_entities = set()
        found_entities = [entity.text.lower() for entity in extract_entities_spacy(text, nlp=nlp).entities]
        
        for triplet in expected_triplets:
            first_ent = re.search('^[^|]*', triplet).group().strip().lower().replace('_', ' ')
            second_ent = re.search('[^|]*$', triplet).group().strip().lower().replace('_', ' ')
            
            all_entities.add(first_ent)
            all_entities.add(second_ent)
        
        compare_pair = [text, list(all_entities), found_entities]
        for_comparison_spacy.append(compare_pair)
    
    return for_comparison_spacy

In [47]:
def evaluate_entity_extraction(for_comparison_data, output_filename=None, threshold=0.95, model=None):
    
    if model is None:
        model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    
    def normalize_text(text):

        if not isinstance(text, str):
            return ""
        text = text.lower().strip()
        text = ' '.join(text.split())
        return text
    
    def extract_number(text):

        match = re.search(r'(\d+[.,]?\d*)', text.replace(',', ''))
        if match:
            try:
                return float(match.group(1))
            except:
                return None
        return None
    
    def normalize_number(text):
   
        num = extract_number(text)
        if num is not None:
            return num
        return None
    
    def is_number_match(str1, str2):
  
        num1 = normalize_number(str1)
        num2 = normalize_number(str2)
        if num1 is not None and num2 is not None:
            return num1 == num2
        return False
    
    def parse_date(date_str):
 
        date_str = date_str.strip()
        
        formats = [
            '%Y-%m-%d',
            '%Y/%m/%d',
            '%d.%m.%Y',
            '%m/%d/%Y',
            '%d/%m/%Y',
            '%B %d, %Y',
            '%b %d, %Y',
            '%d %B %Y',
            '%d %b %Y',
            '%B %d %Y',
            '%b %d %Y',
            '%Y'
        ]
        
        for fmt in formats:
            try:
                return datetime.strptime(date_str, fmt).date()
            except:
                continue
        
        date_patterns = [
            (r'(\d{4})-(\d{1,2})-(\d{1,2})', '%Y-%m-%d'),
            (r'(\d{1,2})/(\d{1,2})/(\d{4})', '%m/%d/%Y'),
            (r'(\d{1,2})\.(\d{1,2})\.(\d{4})', '%d.%m.%Y'),
        ]
        
        for pattern, fmt in date_patterns:
            match = re.search(pattern, date_str)
            if match:
                try:
                    return datetime.strptime(match.group(0), fmt).date()
                except:
                    continue
        
        return None
    
    def is_date_match(str1, str2):
    
        date1 = parse_date(str1)
        date2 = parse_date(str2)
        if date1 and date2:
            return date1 == date2
        return False
    
    def is_substring_match(str1, str2):
     
        str1_norm = normalize_text(str1)
        str2_norm = normalize_text(str2)
        
        if str1_norm in str2_norm or str2_norm in str1_norm:
            min_len = min(len(str1_norm), len(str2_norm))
            if min_len >= 2:
                return True, 1.0
        
        words1 = set(str1_norm.split())
        words2 = set(str2_norm.split())
        
        if words1.issubset(words2) or words2.issubset(words1):
            common_words = words1.intersection(words2)
            if common_words:
                if any(len(w) > 3 for w in common_words):
                    return True, 1.0
                if len(common_words) >= 2:
                    return True, 1.0
        
        return False, 0.0
    
    def is_partial_date_match(str1, str2):

        parts1 = re.findall(r'\d+', str1)
        parts2 = re.findall(r'\d+', str2)
        
        if parts1 and parts2:
            for p1 in parts1:
                for p2 in parts2:
                    if p1 == p2 and len(p1) >= 2:
                        return True
        return False
    
    def evaluate_entities(target, found, threshold=threshold):
        if not target or not found:
            return {
                'correct': 0, 'missed': len(target), 'extra': len(found),
                'correct_list': [], 'missed_list': target, 'extra_list': found,
                'matches': {}, 'total_target': len(target), 'total_found': len(found),
                'precision': 0, 'recall': 0, 'f1': 0
            }
        
        target_embs = model.encode(target, convert_to_tensor=True)
        found_embs = model.encode(found, convert_to_tensor=True)
        
        if target_embs.dim() == 1:
            target_embs = target_embs.unsqueeze(0)
        if found_embs.dim() == 1:
            found_embs = found_embs.unsqueeze(0)
        
        correct = []
        missed = []
        matches = {}
        
        used_found = set()
        
        for i, t in enumerate(target):
            best_score = 0
            best_match = None
            best_match_idx = -1
            match_type = 'none'
            
            for j, f in enumerate(found):
              
                if j in used_found:
                    continue
                
           
                if normalize_text(t) == normalize_text(f):
                    best_score = 1.0
                    best_match = f
                    best_match_idx = j
                    match_type = 'exact'
                    break
            
                if is_date_match(t, f):
                    best_score = 1.0
                    best_match = f
                    best_match_idx = j
                    match_type = 'date'
                    break
         
                if is_number_match(t, f):
                    best_score = 1.0
                    best_match = f
                    best_match_idx = j
                    match_type = 'number'
                    break
       
                is_match, _ = is_substring_match(t, f)
                if is_match:
                    best_score = 1.0
                    best_match = f
                    best_match_idx = j
                    match_type = 'substring'
                    break
             
                if is_partial_date_match(t, f):
                    best_score = 1.0
                    best_match = f
                    best_match_idx = j
                    match_type = 'partial_date'
                    break
                
                score = util.cos_sim(target_embs[i], found_embs[j]).item()
                if score > best_score:
                    best_score = score
                    best_match = f
                    best_match_idx = j
                    match_type = 'cosine'
            
            if best_score >= threshold and best_match_idx != -1:
                correct.append(t)
                matches[t] = {
                    'found': best_match,
                    'score': best_score,
                    'match_type': match_type
                }
                used_found.add(best_match_idx) 
            else:
                missed.append(t)
 
        extra = [f for j, f in enumerate(found) if j not in used_found]
        
        total_target = len(target)
        total_found = len(found)
        total_correct = len(correct)
        
        precision = total_correct / total_found if total_found > 0 else 0
        recall = total_correct / total_target if total_target > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'correct': total_correct,
            'missed': len(missed),
            'extra': len(extra),
            'correct_list': correct,
            'missed_list': missed,
            'extra_list': extra,
            'matches': matches,
            'total_target': total_target,
            'total_found': total_found,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    results = []
    
    for idx, sample in enumerate(for_comparison_data):
        text = sample[0]
        target = sample[1]
        found = sample[2]
        
        result = evaluate_entities(target, found)
        
        results.append({
            'sample_id': idx,
            'text': text,
            'target': target,
            'found': found,
            'precision': result['precision'],
            'recall': result['recall'],
            'f1': result['f1'],
            'correct': result['correct'],
            'missed': result['missed'],
            'extra': result['extra'],
            'correct_list': result['correct_list'],
            'missed_list': result['missed_list'],
            'extra_list': result['extra_list'],
            'matches': result['matches'],
            'total_target': result['total_target'],
            'total_found': result['total_found']
        })
    
    df_results = pd.DataFrame(results)
    
    df_export = pd.DataFrame({
        'Text': df_results['text'],
        'Target Entities': df_results['target'].apply(lambda x: ', '.join(x)),
        'Found Entities': df_results['found'].apply(lambda x: ', '.join(x)),
        'Matched Entities': df_results['correct_list'].apply(lambda x: ', '.join(x)),
        'Missed Entities': df_results['missed_list'].apply(lambda x: ', '.join(x)),
        'Extra Entities': df_results['extra_list'].apply(lambda x: ', '.join(x)),
        'Precision': df_results['precision'],
        'Recall': df_results['recall'],
        'F1': df_results['f1']
    })
    
    if output_filename:
        if not output_filename.endswith('.xlsx'):
            output_filename += '.xlsx'
        df_export.to_excel(output_filename, index=False, engine='openpyxl')
    
    avg_metrics = {
        'precision': df_results['precision'].mean(),
        'recall': df_results['recall'].mean(),
        'f1': df_results['f1'].mean(),
        'total_samples': len(results),
        'total_correct': df_results['correct'].sum(),
        'total_missed': df_results['missed'].sum(),
        'total_extra': df_results['extra'].sum()
    }
    
    return results, df_results, avg_metrics

In [48]:
cmd = 'curl -s "https://datasets-server.huggingface.co/rows?dataset=GEM%2Fweb_nlg&config=en&split=test"'
output = subprocess.check_output(cmd, shell=True, text=True)
data = json.loads(output)

samples = []
for row in data['rows']:
    row_data = row['row']
    samples.append({
        'input': row_data.get('input', []),
        'target': row_data.get('target', '')
    })

model_coref = FCoref(device="cpu")
nlp = spacy.load("en_core_web_sm")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

for_comparison = prepare_comparison_data_with_coref(samples, model=model_coref, nlp=nlp)
results, df_results, avg_metrics = evaluate_entity_extraction(
    for_comparison, 
    output_filename='results_with_coref.xlsx', 
    threshold=0.95,
    model=embedding_model
)

for_comparison_spacy = prepare_comparison_data_spacy(samples, nlp=nlp)
results_spacy, df_results_spacy, avg_metrics_spacy = evaluate_entity_extraction(
    for_comparison_spacy, 
    output_filename='results_spacy_only.xlsx', 
    threshold=0.95,
    model=embedding_model
)

summary_df = pd.DataFrame({
    'Method': ['With Coreference', 'SpaCy Only'],
    'Precision': [avg_metrics['precision'], avg_metrics_spacy['precision']],
    'Recall': [avg_metrics['recall'], avg_metrics_spacy['recall']],
    'F1': [avg_metrics['f1'], avg_metrics_spacy['f1']]
})
summary_df.to_excel('metrics_summary.xlsx', index=False)

07/19/2026 15:53:27 - INFO - 	 missing_keys: []
07/19/2026 15:53:27 - INFO - 	 unexpected_keys: []
07/19/2026 15:53:27 - INFO - 	 mismatched_keys: []
07/19/2026 15:53:27 - INFO - 	 error_msgs: []
07/19/2026 15:53:27 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M
07/19/2026 15:53:30 - INFO - 	 Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
07/19/2026 15:53:34 - INFO - 	 Tokenize 1 inputs...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

c:\Python311\Lib\site-packages\pyarrow\compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
07/19/2026 15:53:34 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

07/19/2026 15:53:34 - INFO - 	 Tokenize 1 inputs...


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]